In [1]:
import os
import glob
import pandas as pd
from pathlib import Path

In [4]:
BASE_DIR = "cleanconll"  # ajuste se necessário
REPORTS_DIR = os.path.join(BASE_DIR, "reports_out")
Path(REPORTS_DIR).mkdir(parents=True, exist_ok=True)


# util simples para listar subpastas imediatas
def list_subdirs(path):
    return sorted([p for p in os.listdir(path) if (Path(path) / p).is_dir()])

In [5]:
INS_DIR = os.path.join(BASE_DIR, "insights_out")
splits_insights = list_subdirs(INS_DIR)

In [6]:
def load_insights(base_dir):
    out = {}
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)
        try:
            q = pd.read_csv(os.path.join(sdir, "length_quantiles.csv"))
            q.insert(0, "split", split)
        except FileNotFoundError:
            q = None
        try:
            oov = pd.read_csv(os.path.join(sdir, "oov_rates.csv"))
            oov.insert(0, "split", split)
        except FileNotFoundError:
            oov = None
        try:
            rare = pd.read_csv(os.path.join(sdir, "rare_labels.csv"))
            rare.insert(0, "split", split)
        except FileNotFoundError:
            rare = None
        try:
            tok_te = pd.read_csv(os.path.join(sdir, "tokens_test.csv"))
            tok_te.insert(0, "split", split)
        except FileNotFoundError:
            tok_te = None
        try:
            tok_tr = pd.read_csv(os.path.join(sdir, "tokens_train.csv"))
            tok_tr.insert(0, "split", split)
        except FileNotFoundError:
            tok_tr = None

        out[split] = {
            "quantiles": q,
            "oov": oov,
            "rare": rare,
            "tokens_test": tok_te,
            "tokens_train": tok_tr,
        }
    return out

In [7]:
ins = load_insights(INS_DIR)

In [8]:
# Tabelas consolidadas simples:
ins_quantiles = pd.concat(
    [ins[s]["quantiles"] for s in ins if ins[s]["quantiles"] is not None],
    ignore_index=True,
)
ins_oov = pd.concat(
    [ins[s]["oov"] for s in ins if ins[s]["oov"] is not None], ignore_index=True
)
ins_rare = pd.concat(
    [ins[s]["rare"] for s in ins if ins[s]["rare"] is not None], ignore_index=True
)
ins_tok_te = pd.concat(
    [ins[s]["tokens_test"] for s in ins if ins[s]["tokens_test"] is not None],
    ignore_index=True,
)
ins_tok_tr = pd.concat(
    [ins[s]["tokens_train"] for s in ins if ins[s]["tokens_train"] is not None],
    ignore_index=True,
)

# OOV em formato largo por métrica
ins_oov_wide = ins_oov.pivot(
    index="split", columns="metric", values="value"
).reset_index()

In [9]:
display(ins_oov_wide)

metric,split,oov_rate_test_vs_train,oov_rate_val_vs_train
0,adversarial,0.092271,0.059514
1,heur_len,0.064721,0.062181
2,heur_rare,0.109650,0.047378
3,loc,0.096048,0.100807
4,reverse,0.086601,0.076101
5,semantic,0.113788,0.073079
6,standard,0.058656,0.056937


In [10]:
CD_DIR = os.path.join(BASE_DIR, "class_dist_out")


def load_class_dist(base_dir):
    rows_counts, rows_props, rows_long = [], [], []
    label_vecs_all = []
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)

        # long
        if Path(os.path.join(sdir, "class_distribution_long.csv")).exists():
            df_long = pd.read_csv(os.path.join(sdir, "class_distribution_long.csv"))
            df_long.insert(0, "split_name", split)
            rows_long.append(df_long)

        # counts e props
        if Path(os.path.join(sdir, "counts_pivot.csv")).exists():
            cnt = pd.read_csv(os.path.join(sdir, "counts_pivot.csv"))
            cnt.insert(0, "split_name", split)
            rows_counts.append(cnt)

        if Path(os.path.join(sdir, "props_pivot.csv")).exists():
            pr = pd.read_csv(os.path.join(sdir, "props_pivot.csv"))
            pr.insert(0, "split_name", split)
            rows_props.append(pr)

        # label_vecs.csv (linha por part)
        if Path(os.path.join(sdir, "label_vecs.csv")).exists():
            lv = pd.read_csv(os.path.join(sdir, "label_vecs.csv"))
            lv.insert(0, "split_name", split)
            label_vecs_all.append(lv)

    long_df = pd.concat(rows_long, ignore_index=True) if rows_long else pd.DataFrame()
    counts = (
        pd.concat(rows_counts, ignore_index=True) if rows_counts else pd.DataFrame()
    )
    props = pd.concat(rows_props, ignore_index=True) if rows_props else pd.DataFrame()
    lvecs = (
        pd.concat(label_vecs_all, ignore_index=True)
        if label_vecs_all
        else pd.DataFrame()
    )
    return long_df, counts, props, lvecs


cd_long, cd_counts, cd_props, cd_lv = load_class_dist(CD_DIR)

In [11]:
merged_wide = cd_counts.merge(
    cd_props,
    on=["split_name", "label"],
    suffixes=("_count", "_prop"),
)
merged_wide.to_csv(
    os.path.join(REPORTS_DIR, "class_counts_props_merged_wide.csv"), index=False
)

In [12]:
props_long = cd_props.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="prop",
)
props_long_sorted = props_long.sort_values(
    ["split_name", "part", "prop"], ascending=[True, True, False]
)

# counts longo para anexar contagem ao top-5
counts_long = cd_counts.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="count",
)

In [13]:
top5 = (
    props_long_sorted.groupby(["split_name", "part"], group_keys=False)
    .head(5)
    .merge(counts_long, on=["split_name", "label", "part"], how="left")
)

In [14]:
print("== Merged (wide) ==")
display(merged_wide.head())

== Merged (wide) ==


,split_name,label,test_count,train_count,val_count,test_prop,train_prop,val_prop
0,adversarial,B-LOC,1615,6763,1021,0.030891,0.031086,0.032201
1,adversarial,B-MISC,857,3977,585,0.016392,0.018280,0.018450
2,adversarial,B-ORG,2410,7004,1078,0.046097,0.032194,0.033999
3,adversarial,B-PER,1676,7216,1055,0.032058,0.033168,0.033273
4,adversarial,I-LOC,160,1022,193,0.003060,0.004698,0.006087


In [15]:
print("\n== Props (long, sorted) ==")
display(props_long_sorted.head(12))


== Props (long, sorted) ==


,split_name,label,part,prop
8,adversarial,O,test,0.819552
2,adversarial,B-ORG,test,0.046097
3,adversarial,B-PER,test,0.032058
0,adversarial,B-LOC,test,0.030891
7,adversarial,I-PER,test,0.025172
6,adversarial,I-ORG,test,0.020543
1,adversarial,B-MISC,test,0.016392
5,adversarial,I-MISC,test,0.006236
4,adversarial,I-LOC,test,0.003060
71,adversarial,O,train,0.833316


In [16]:
print("\n== Top-5 por split e partição ==")
display(top5)


== Top-5 por split e partição ==


,split_name,label,part,prop,count
0,adversarial,O,test,0.819552,42847
1,adversarial,B-ORG,test,0.046097,2410
2,adversarial,B-PER,test,0.032058,1676
3,adversarial,B-LOC,test,0.030891,1615
4,adversarial,I-PER,test,0.025172,1316
...,...,...,...,...,...
100,standard,O,val,0.829222,25414
101,standard,B-ORG,val,0.037327,1144
102,standard,B-PER,val,0.032172,986
103,standard,B-LOC,val,0.030573,937


In [17]:
top5['part'].value_counts()

part
test     35
train    35
val      35
Name: count, dtype: int64

In [18]:
parts = ["train", "val", "test"]

for split in sorted(top5["split_name"].unique()):
    print(f"\n=== {split} ===")
    for part in parts:
        sub = (
            top5[(top5["split_name"] == split) & (top5["part"] == part)]
            .sort_values("prop", ascending=False)
            .head(5)
            .loc[:, ["label", "prop", "count"]]
            .reset_index(drop=True)
        )
        print(f"\n[{part}] top-5")
        try:
            display(sub)  # funciona no Jupyter
        except NameError:
            print(sub.to_string(index=False))  # fallback se display não existir


=== adversarial ===

[train] top-5


,label,prop,count
0,O,0.833316,181293
1,B-PER,0.033168,7216
2,B-ORG,0.032194,7004
3,B-LOC,0.031086,6763
4,I-PER,0.023065,5018



[val] top-5


,label,prop,count
0,O,0.824045,26128
1,B-ORG,0.033999,1078
2,B-PER,0.033273,1055
3,B-LOC,0.032201,1021
4,I-PER,0.022866,725



[test] top-5


...

,label,prop,count
0,O,0.831079,200270
1,B-ORG,0.034182,8237
2,B-PER,0.032970,7945
3,B-LOC,0.031273,7536
4,I-PER,0.023434,5647



[val] top-5


,label,prop,count
0,O,0.829222,25414
1,B-ORG,0.037327,1144
2,B-PER,0.032172,986
3,B-LOC,0.030573,937
4,I-PER,0.022155,679



[test] top-5


,label,prop,count
0,O,0.821658,24584
1,B-ORG,0.037132,1111
2,B-PER,0.033957,1016
3,B-LOC,0.030949,926
4,I-PER,0.024499,733


In [19]:
CWI_DIR = os.path.join(BASE_DIR, "cosine_out")

# tenta usar o resumo pronto; se não existir, empilha de cada split
summary_path = os.path.join(CWI_DIR, "cosine_summary_all_splits.csv")
if Path(summary_path).exists():
    cos_within_all = pd.read_csv(summary_path)
else:
    rows = []
    for split in list_subdirs(CWI_DIR):
        f = os.path.join(CWI_DIR, split, "cosine_all.csv")
        if Path(f).exists():
            df = pd.read_csv(f)
            df.insert(0, "split", split)
            rows.append(df)
    cos_within_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

# Tabelas simples:
# - matriz (a,b) por espaço+split_set (labels/words e pares)
if not cos_within_all.empty:
    cos_within_pivot = (
        cos_within_all.assign(pair=lambda d: d["a"] + "_" + d["b"])
        .pivot_table(
            index=["split", "space"],
            columns="pair",
            values="cosine_distance",
            aggfunc="first",
        )
        .reset_index()
    )
else:
    cos_within_pivot = pd.DataFrame()

In [20]:
cos_within_all.query("space == 'words'")[['split', 'a', 'b', 'cosine_distance']]

,split,a,b,cosine_distance
3,standard,train,val,0.043638
4,standard,train,test,0.038089
5,standard,val,test,0.069665
9,heur_len,train,val,0.038864
10,heur_len,train,test,0.022370
11,heur_len,val,test,0.050601
15,heur_rare,train,val,0.038871
16,heur_rare,train,test,0.128146
17,heur_rare,val,test,0.125565
21,adversarial,train,val,0.038356


In [21]:
cos_within_all.query("space == 'labels'")[["split", "a", "b", "cosine_distance"]]

,split,a,b,cosine_distance
0,standard,train,val,0.000012
1,standard,train,test,0.000018
2,standard,val,test,0.000014
6,heur_len,train,val,0.000017
7,heur_len,train,test,0.000009
8,heur_len,val,test,0.000012
12,heur_rare,train,val,0.000015
13,heur_rare,train,test,0.000157
14,heur_rare,val,test,0.000236
18,adversarial,train,val,0.000020


In [22]:
display(cos_within_pivot)

pair,split,space,train_test,train_val,val_test
0,adversarial,labels,0.000170,0.000020,0.000129
1,adversarial,words,0.180149,0.038356,0.187341
2,heur_len,labels,0.000009,0.000017,0.000012
3,heur_len,words,0.022370,0.038864,0.050601
4,heur_rare,labels,0.000157,0.000015,0.000236
5,heur_rare,words,0.128146,0.038871,0.125565
6,loc,labels,0.000244,0.000074,0.000208
7,loc,words,0.111601,0.187837,0.141729
8,reverse,labels,0.001749,0.001586,0.001519
9,reverse,words,0.150017,0.514313,0.499412


In [23]:
cos_within_pivot.query("space == 'labels'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
0,adversarial,0.000170,0.000020,0.000129
2,heur_len,0.000009,0.000017,0.000012
4,heur_rare,0.000157,0.000015,0.000236
6,loc,0.000244,0.000074,0.000208
8,reverse,0.001749,0.001586,0.001519
10,semantic,0.004264,0.000609,0.001950
12,standard,0.000018,0.000012,0.000014


In [24]:
cos_within_pivot.query("space == 'words'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
1,adversarial,0.180149,0.038356,0.187341
3,heur_len,0.022370,0.038864,0.050601
5,heur_rare,0.128146,0.038871,0.125565
7,loc,0.111601,0.187837,0.141729
9,reverse,0.150017,0.514313,0.499412
11,semantic,0.664695,0.309630,0.678717
13,standard,0.038089,0.043638,0.069665


In [25]:
CB_DIR = os.path.join(BASE_DIR, "cosine_between_out")


def safe_read_csv(path):
    return pd.read_csv(path) if Path(path).exists() else None


cos_bw_train_words = safe_read_csv(os.path.join(CB_DIR, "cos_train_words.csv"))
cos_bw_test_words = safe_read_csv(os.path.join(CB_DIR, "cos_test_words.csv"))
cos_bw_train_labels = safe_read_csv(os.path.join(CB_DIR, "cos_train_labels.csv"))
cos_bw_test_labels = safe_read_csv(os.path.join(CB_DIR, "cos_test_labels.csv"))

In [26]:
cos_bw_train_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.001103,0.011596,0.010730,0.012317,0.083228,0.066885
1,heur_len,0.001103,0.000000,0.013544,0.011574,0.013983,0.081402,0.066263
2,heur_rare,0.011596,0.013544,0.000000,0.023985,0.008501,0.135609,0.097801
3,adversarial,0.010730,0.011574,0.023985,0.000000,0.021160,0.072983,0.074994
4,loc,0.012317,0.013983,0.008501,0.021160,0.000000,0.130122,0.098370
5,semantic,0.083228,0.081402,0.135609,0.072983,0.130122,0.000000,0.067984
6,reverse,0.066885,0.066263,0.097801,0.074994,0.098370,0.067984,0.000000


In [27]:
cos_bw_test_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.048559,0.187885,0.176221,0.143895,0.355242,0.173313
1,heur_len,0.048559,0.000000,0.152562,0.160721,0.112591,0.351299,0.144932
2,heur_rare,0.187885,0.152562,0.000000,0.301944,0.085854,0.734999,0.150660
3,adversarial,0.176221,0.160721,0.301944,0.000000,0.262409,0.329528,0.290379
4,loc,0.143895,0.112591,0.085854,0.262409,0.000000,0.594886,0.181501
5,semantic,0.355242,0.351299,0.734999,0.329528,0.594886,0.000000,0.535333
6,reverse,0.173313,0.144932,0.150660,0.290379,0.181501,0.535333,0.000000


In [28]:
cos_bw_train_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.913768,0.909044,0.000004,0.908529,0.000546,0.902558
1,heur_len,0.913768,0.000000,0.000400,0.916263,0.000456,0.920432,0.000240
2,heur_rare,0.909044,0.000400,0.000000,0.911520,0.000007,0.914832,0.000662
3,adversarial,0.000004,0.916263,0.911520,0.000000,0.911004,0.000512,0.905055
4,loc,0.908529,0.000456,0.000007,0.911004,0.000000,0.914261,0.000744
5,semantic,0.000546,0.920432,0.914832,0.000512,0.914261,0.000000,0.909207
6,reverse,0.902558,0.000240,0.000662,0.905055,0.000744,0.909207,0.000000


In [29]:
cos_bw_test_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.905649,0.920585,0.000074,0.920593,0.002999,0.925275
1,heur_len,0.905649,0.000000,0.000628,0.894874,0.000461,0.852886,0.000788
2,heur_rare,0.920585,0.000628,0.000000,0.909863,0.000083,0.867758,0.001009
3,adversarial,0.000074,0.894874,0.909863,0.000000,0.909815,0.002511,0.914635
4,loc,0.920593,0.000461,0.000083,0.909815,0.000000,0.867730,0.001006
5,semantic,0.002999,0.852886,0.867758,0.002511,0.867730,0.000000,0.871660
6,reverse,0.925275,0.000788,0.001009,0.914635,0.001006,0.871660,0.000000
